# 05 — Things single-cell RNA-seq cannot do

**Day 2, 11:30–12:15**

This is the notebook the whole workshop is built around. Three analyses, each of which
has **no possible scRNA-seq equivalent** — not "harder", not "less powerful".
Impossible, because the measurement was thrown away at dissociation.

1. **Distance to an anatomical structure** as a continuous covariate — including an
   axis you draw yourself
2. **Direct cell–cell contact** with a spatially correct null
3. **Segmentation-free signal** — what is there before anyone drew a polygon

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import seaborn as sns
from scipy.spatial import cKDTree

sc.settings.verbosity = 1
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
NAVY, GOLD, CORAL, ICE = "#001158", "#FBAE40", "#F26B43", "#BCD2FF"

adata = sc.read_h5ad(DATA / "ovarian_niches.h5ad")   # from notebook 04
x, y = adata.obsm["spatial"].T
types = list(adata.obs["cell_type"].cat.categories)
types

## 1. Distance to the tumour boundary

In scRNA-seq, "distance from the tumour edge" does not exist as a variable. Here it
is just geometry, and it turns a categorical comparison into a **continuous dose–response**.

The recipe: pick a reference population, build a KD-tree of its coordinates, and ask
every other cell how far away it is.

In [ ]:
def biggest_matching(*patterns, default_idx=0):
    """The most abundant cell type whose name contains any of these strings.

    Picking the *largest* match matters now that the annotation has several
    tumour and several fibroblast subtypes: an alphabetical first-match would
    happily choose a rare proliferating subcluster as the tumour reference.
    """
    counts = adata.obs["cell_type"].value_counts()
    hits = [t for t in counts.index
            if any(p.lower() in str(t).lower() for p in patterns)]
    return hits[0] if hits else types[default_idx]


TUMOUR = biggest_matching("tumour", "epithel")
print("reference population:", TUMOUR)

is_tum = (adata.obs["cell_type"] == TUMOUR).to_numpy()
tree = cKDTree(adata.obsm["spatial"][is_tum])
dist, _ = tree.query(adata.obsm["spatial"], k=1)
adata.obs["dist_to_tumour"] = dist
adata.obs.loc[is_tum, "dist_to_tumour"] = 0.0

print(adata.obs.loc[~is_tum, "dist_to_tumour"].describe().round(1))

> **A caution before the pretty plot.** Nearest-neighbour distance to a *cell type*
> is not the same as distance to a *structure*. A single misclassified tumour cell
> sitting alone in the stroma creates a false "tumour" 200 um from any nest, and
> every cell around it gets a distance of ~10 um. Two defences: require a minimum
> local density of tumour cells before a point counts as tumour, or build the
> reference from your **niche** labels (notebook 04) instead of cell types, since
> niches are already smoothed over neighbourhoods.

In [ ]:
# more robust reference: tumour cells that are themselves in a tumour-dominated niche
tum_niche = (
    adata.obs.groupby("niche", observed=True)
    .apply(lambda d: (d["cell_type"] == TUMOUR).mean())
    .idxmax()
)
core = is_tum & (adata.obs["niche"] == tum_niche).to_numpy()
print(f"tumour-dominated niche: {tum_niche}  ({core.sum():,} reference cells)")

tree = cKDTree(adata.obsm["spatial"][core])
d2, _ = tree.query(adata.obsm["spatial"], k=1)
adata.obs["dist_to_nest"] = d2

fig, axes = plt.subplots(1, 2, figsize=(13, 5.6))
pc = axes[0].scatter(x, y, c=np.clip(d2, 0, 300), s=1.1, cmap="viridis",
                     linewidths=0, rasterized=True)
plt.colorbar(pc, ax=axes[0], label="distance to tumour nest (um)")
axes[0].set_title("distance field")
axes[1].scatter(x, y, s=0.6, c="0.88", linewidths=0, rasterized=True)
axes[1].scatter(x[core], y[core], s=1.6, c=NAVY, linewidths=0, rasterized=True)
axes[1].set_title("reference population")
for ax in axes:
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

In [ ]:
# Composition as a function of distance — the invasive front, quantified
bins = [0, 10, 25, 50, 100, 200, 400, np.inf]
labels = ["0-10", "10-25", "25-50", "50-100", "100-200", "200-400", ">400"]
adata.obs["dist_bin"] = pd.cut(adata.obs["dist_to_nest"], bins=bins, labels=labels)

frac = (
    pd.crosstab(adata.obs["dist_bin"], adata.obs["cell_type"], normalize="index")
    .drop(columns=[TUMOUR], errors="ignore")
)
fig, ax = plt.subplots(figsize=(8, 4.6))
frac.plot(kind="bar", stacked=True, ax=ax, colormap="tab20", width=0.85)
ax.set_xlabel("distance from tumour nest (um)"); ax.set_ylabel("fraction of cells")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=8)
ax.set_title("tissue composition as a function of distance")
plt.tight_layout(); plt.show()

> **Try it yourself — how far does the effect reach?**
>
> Pick a cell type and see how its abundance changes with distance from the tumour.
> One number per distance band, so you can read the trend without interpreting a
> stacked bar chart.

In [ ]:
CELL_TYPE = adata.obs["cell_type"].value_counts().index[1]    # <-- CHANGE THIS

print(f"cell types available:\n{list(adata.obs['cell_type'].cat.categories)}\n")
print(f"showing: {CELL_TYPE}\n")

tab = pd.crosstab(adata.obs["dist_bin"], adata.obs["cell_type"], normalize="index")
if CELL_TYPE not in tab.columns:
    raise KeyError(f"{CELL_TYPE!r} is not one of the cell types listed above")
frac_by_band = tab[CELL_TYPE]

for band, frac in frac_by_band.items():
    bar = "#" * int(round(100 * frac / max(frac_by_band.max(), 1e-9) * 0.4))
    print(f"  {str(band):>9} um   {100 * frac:5.1f}%  {bar}")

In [ ]:
# Gene expression as a function of distance, within ONE cell type.
# This is the design that has no scRNA-seq equivalent: same cell type,
# different position, so position is the only variable.
CELLTYPE = biggest_matching("fibro", "caf", default_idx=min(1, len(types) - 1))
sub = adata[(adata.obs["cell_type"] == CELLTYPE) & (adata.obs["dist_to_nest"] < 400)].copy()
print(f"{CELLTYPE}: {sub.n_obs:,} cells within 400 um of a nest")

# genes that actually vary among the fibroblasts in this section
genes = [g for g in ["POSTN", "COL1A1", "DCN", "LUM", "TIMP3", "MFAP5", "C7"]
         if g in sub.var_names][:4]
X = sub[:, genes].X
X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)

fig, axes = plt.subplots(1, len(genes), figsize=(4.0 * len(genes), 3.6), sharex=True)
axes = np.atleast_1d(axes)
edges = np.arange(0, 401, 25)
centres = 0.5 * (edges[:-1] + edges[1:])
which = np.digitize(sub.obs["dist_to_nest"], edges) - 1
for k, (g, ax) in enumerate(zip(genes, axes)):
    means = [X[which == i, k].mean() if (which == i).sum() > 20 else np.nan
             for i in range(len(centres))]
    sems = [X[which == i, k].std() / max(np.sqrt((which == i).sum()), 1)
            if (which == i).sum() > 20 else np.nan for i in range(len(centres))]
    means, sems = np.array(means), np.array(sems)
    ax.plot(centres, means, color=NAVY, lw=2)
    ax.fill_between(centres, means - 1.96 * sems, means + 1.96 * sems, color=ICE, alpha=0.6)
    ax.set_title(g); ax.set_xlabel("distance to nest (um)")
axes[0].set_ylabel(f"mean expression\n({CELLTYPE})")
sns.despine(); plt.tight_layout(); plt.show()

A sloping line here is a **spatial gradient of cell state within one cell type**.
If you had dissociated this tissue, every one of those cells would have collapsed
into a single point in the fibroblast cluster and the gradient would be invisible —
you might have called it "CAF heterogeneity" and gone looking for subclusters.

### Exercise 5.1
Do the same for an immune population, and for distance to **vessels** instead of
tumour (use the endothelial cells as reference). Does any gene respond to one
distance and not the other? Watch out: the two distances are correlated, so
condition on both before claiming either.

In [ ]:
# your code here

### 1b. Draw your own axis

Distance to a cell type works when the structure *is* a cell type. Often it is not:
a capsule, a lumen, a duct, the edge of a lesion, the boundary between two lobes.
None of those has a marker, but you can see them.

So draw the line yourself. Pick two points, and every cell gets a **signed
perpendicular distance** to the line through them — negative on one side, positive on
the other. That single number then does everything section 1 did, but for a structure
you defined by eye.

This is the general form of the analysis. Distance to a nest is one special case of
it; a tissue boundary that no gene marks is another.

In [ ]:
# A reference map with coordinates on, so you can read two points off it.
fig, ax = plt.subplots(figsize=(8, 8))
cats = adata.obs["cell_type"].astype("category")
colours = adata.uns.get("cell_type_colors")
if colours is None or len(colours) != len(cats.cat.categories):
    cm = plt.get_cmap("tab20")
    colours = [cm(i % 20) for i in range(len(cats.cat.categories))]
for colour, name in zip(colours, cats.cat.categories):
    m = (cats == name).to_numpy()
    ax.scatter(x[m], y[m], s=1.0, color=colour, label=name, linewidths=0, rasterized=True)
ax.set_aspect("equal"); ax.invert_yaxis()
ax.grid(alpha=0.3, linestyle=":")
ax.set_xlabel("x (µm)"); ax.set_ylabel("y (µm)")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False,
          markerscale=9, fontsize=8)
ax.set_title("read two points off this grid")
plt.tight_layout(); plt.show()

print(f"x runs {x.min():.0f} to {x.max():.0f} um,  y runs {y.min():.0f} to {y.max():.0f} um")

In [ ]:
# ---- YOUR AXIS -----------------------------------------------------------
# Two points defining the line. Read them off the map above. The line is
# extended infinitely in both directions — the points only fix its position
# and angle, they are not endpoints.
P1 = (x.min() + 0.30 * (x.max() - x.min()), y.min())                 # <-- CHANGE
P2 = (x.min() + 0.55 * (x.max() - x.min()), y.max())                 # <-- CHANGE
# --------------------------------------------------------------------------


def signed_distance_to_line(coords, p1, p2):
    """Perpendicular distance from each point to the line through p1 and p2.

    The sign says which side you are on: the cross product of (p2 - p1) with
    (point - p1) is positive on one side and negative on the other. Dividing by
    the length of (p2 - p1) turns it into a distance in microns.
    """
    (x1, y1), (x2, y2) = p1, p2
    dx, dy = x2 - x1, y2 - y1
    length = np.hypot(dx, dy)
    if length == 0:
        raise ValueError("P1 and P2 are the same point — the line is undefined")
    return ((coords[:, 0] - x1) * dy - (coords[:, 1] - y1) * dx) / length


adata.obs["axis_distance"] = signed_distance_to_line(adata.obsm["spatial"], P1, P2)
d = adata.obs["axis_distance"]
print(f"signed distance runs {d.min():.0f} to {d.max():.0f} um")
print(f"{(d < 0).sum():,} cells on the negative side, {(d > 0).sum():,} on the positive side")

In [ ]:
# Check the line is where you meant it to be BEFORE interpreting anything.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

lim = np.percentile(np.abs(d), 98)
pc = axes[0].scatter(x, y, c=np.clip(d, -lim, lim), s=1.1, cmap="RdBu_r",
                     vmin=-lim, vmax=lim, linewidths=0, rasterized=True)
plt.colorbar(pc, ax=axes[0], label="signed distance to your axis (µm)")
axes[0].set_title("the distance field")

for colour, name in zip(colours, cats.cat.categories):
    m = (cats == name).to_numpy()
    axes[1].scatter(x[m], y[m], s=1.0, color=colour, linewidths=0, rasterized=True)
axes[1].set_title("cell types, same view")

# draw the line across both panels
for ax in axes:
    (x1, y1), (x2, y2) = P1, P2
    dx, dy = x2 - x1, y2 - y1
    t = np.array([-5, 5])                       # extend well past the tissue
    ax.plot(x1 + t * dx, y1 + t * dy, color="k", lw=2, ls="--")
    ax.scatter(*zip(P1, P2), color="k", s=45, zorder=5)
    ax.set_xlim(x.min(), x.max()); ax.set_ylim(y.max(), y.min())
    ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()

### The same analysis as section 1, on your axis

Two outputs, matching what we produced for distance-to-nest: composition by distance
band, and expression against distance within one cell type. The difference is that
your axis has **two sides**, so the plots run from negative to positive rather than
from zero outwards — which makes an asymmetry visible if there is one.

In [ ]:
BAND = 100.0        # µm per band  <-- CHANGE THIS

edges = np.arange(np.floor(d.min() / BAND) * BAND,
                  np.ceil(d.max() / BAND) * BAND + BAND, BAND)
adata.obs["axis_bin"] = pd.cut(d, bins=edges)

frac = pd.crosstab(adata.obs["axis_bin"], adata.obs["cell_type"], normalize="index")
frac = frac.loc[(pd.crosstab(adata.obs["axis_bin"], adata.obs["cell_type"]).sum(axis=1) >= 50)]

fig, ax = plt.subplots(figsize=(9, 4.6))
frac.plot(kind="bar", stacked=True, ax=ax, width=0.9,
          color=[dict(zip(cats.cat.categories, colours)).get(c) for c in frac.columns])
ax.set_xlabel("signed distance to your axis (µm)")
ax.set_ylabel("fraction of cells")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), frameon=False, fontsize=8)
ax.set_title("composition across your axis")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

In [ ]:
# Expression against signed distance, within one cell type — the same design as
# section 1, and still the one with no scRNA-seq equivalent: same cell type,
# different position, so position is the only variable.
CELL_TYPE = biggest_matching("fibro", "caf")     # <-- CHANGE THIS
GENES = ["POSTN", "COL4A1", "DCN", "TIMP3"]      # <-- CHANGE THIS

sub = adata[adata.obs["cell_type"] == CELL_TYPE].copy()
genes = [g for g in GENES if g in sub.var_names][:4]
print(f"{CELL_TYPE}: {sub.n_obs:,} cells; plotting {genes}")

X = sub[:, genes].X
X = X.toarray() if sp.issparse(X) else np.asarray(X)
dd = sub.obs["axis_distance"].to_numpy()

centres = 0.5 * (edges[:-1] + edges[1:])
which = np.digitize(dd, edges) - 1

fig, axes = plt.subplots(1, len(genes), figsize=(4.0 * len(genes), 3.6), sharex=True)
for k, (g, ax) in enumerate(zip(genes, np.atleast_1d(axes))):
    means, sems = [], []
    for i in range(len(centres)):
        v = X[which == i, k]
        means.append(v.mean() if len(v) > 20 else np.nan)
        sems.append(v.std() / np.sqrt(len(v)) if len(v) > 20 else np.nan)
    means, sems = np.array(means), np.array(sems)
    ax.plot(centres, means, color=NAVY, lw=2)
    ax.fill_between(centres, means - 1.96 * sems, means + 1.96 * sems,
                    color=ICE, alpha=0.6)
    ax.axvline(0, color="k", lw=1.2, ls="--")
    ax.set_title(g); ax.set_xlabel("signed distance (µm)")
np.atleast_1d(axes)[0].set_ylabel(f"mean expression\n({CELL_TYPE})")
sns.despine(); plt.tight_layout(); plt.show()

### Reading it

The dashed line at zero is your axis. A gene that rises on one side and not the other
is responding to something about that side — which is a stronger and more specific
claim than "expression varies with distance from the tumour", because you chose the
boundary and can say what it is anatomically.

Three cautions, in order of how often they matter:

1. **A straight line is a model.** If the structure curves, cells near the ends of your
   line get distances that mean something different from cells near the middle. For a
   curved front, use the niche-based distance from section 1 instead.
2. **You drew it, so you can draw it to get the answer you want.** Fix the line
   *before* you look at the expression plots, and say in the methods how you placed it.
3. **Check the cell counts per band.** Bands at the extremes often hold very few cells,
   and the confidence ribbon will tell you — it widens where the data thins.

### Exercise 5.1b
Place your axis along a boundary you can see but cannot name with a marker — the edge
of the section, a fold, the interface between two stromal regions. Then find one gene
that responds to it. If nothing does, that is a real result: the boundary is
architectural, not transcriptional.

In [ ]:
# your code here

## 2. Cell–cell contact, with a null that respects geometry

"Cell type A talks to cell type B" is inferred in scRNA-seq from ligand and receptor
expression in two clusters — with no evidence the cells were ever within a millimetre
of each other. Here you can require **actual contact**.

The null model is the delicate part. Shuffling labels at random destroys tissue
structure and makes everything look significant. A better null keeps each cell's
neighbourhood and shuffles labels **within distance-matched strata**, or permutes
labels only among cells of similar local density. Below we use the simplest defensible
version: permute labels within niche, so the null preserves large-scale architecture
and only tests fine-scale arrangement.

In [ ]:
A = adata.obsp["spatial_connectivities"].tocsr()
labels = adata.obs["cell_type"].to_numpy()
niches = adata.obs["niche"].to_numpy()

def contact_counts(lab):
    oh = pd.get_dummies(pd.Categorical(lab, categories=types)).to_numpy().astype(float)
    return oh.T @ (A @ oh)

obs_counts = contact_counts(labels)

rng = np.random.default_rng(0)
N_PERM = 200
null = np.zeros((N_PERM, len(types), len(types)))
for p in range(N_PERM):
    perm = labels.copy()
    for n in np.unique(niches):                    # shuffle WITHIN niche
        m = niches == n
        perm[m] = rng.permutation(perm[m])
    null[p] = contact_counts(perm)

z = (obs_counts - null.mean(0)) / (null.std(0) + 1e-9)
z_df = pd.DataFrame(z, index=types, columns=types)

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(z_df, cmap="RdBu_r", center=0, annot=True, fmt=".0f", ax=ax,
            cbar_kws={"label": "z vs within-niche null"})
ax.set_title("contact enrichment, architecture-preserving null")
plt.tight_layout(); plt.show()

Compare this with `sq.gr.nhood_enrichment` from notebook 04. **The z-scores are much
smaller.** The global-shuffle null was crediting the method for structure you could
already see with your eyes; this one only reports arrangement beyond the niche level.

Whenever a spatial result looks spectacular, ask what the null was. Most of the
time, the null was too easy to beat.

In [ ]:
# Ligand-receptor, spatially constrained: does a specific L-R pair sit across contacts
# more than expected? Self-contained; no database download needed.
PAIR = ("CXCL12", "CXCR4")
if all(g in adata.var_names for g in PAIR):
    L = np.asarray(adata[:, PAIR[0]].X.todense()).ravel()
    R = np.asarray(adata[:, PAIR[1]].X.todense()).ravel()
    lpos, rpos = L > np.percentile(L, 90), R > np.percentile(R, 90)

    obs_edges = float(lpos @ (A @ rpos))
    perm_edges = []
    for _ in range(500):
        pr = rng.permutation(rpos)
        perm_edges.append(float(lpos @ (A @ pr)))
    perm_edges = np.array(perm_edges)
    zz = (obs_edges - perm_edges.mean()) / perm_edges.std()
    print(f"{PAIR[0]}-high  ->  {PAIR[1]}-high contacts")
    print(f"  observed {obs_edges:,.0f}   null {perm_edges.mean():,.0f} +/- {perm_edges.std():,.0f}")
    print(f"  z = {zz:.1f}")
else:
    print(f"{PAIR} not both on the panel — pick another pair present in adata.var_names")

> **Optional:** `sq.gr.ligrec()` runs this systematically against the OmniPath
> database, but it downloads the database on first use. If the workshop wifi
> cooperates, try it; if not, the hand-rolled version above is the same idea and you
> can see every moving part.

## 3. Before anyone drew a polygon

Every result so far rests on the segmentation. Here is how to check whether a
finding survives without it: rasterise the transcripts onto a grid and analyse the
grid. No cells, no polygons, no assumptions.

In [ ]:
# The transcript table, loaded here because this section works from molecules
# rather than cells. qv is the decoding confidence 10x assigns each molecule; 20
# is their own threshold, and the count matrix is already filtered at it, but the
# transcript table is not.
tx = pd.read_parquet(DATA / "transcripts_crop.parquet")
tx = tx[tx["qv"] >= 20]
print(f"{len(tx):,} high-confidence transcripts")

BIN = 10  # um
gx = ((tx["x_location"] - tx["x_location"].min()) // BIN).astype(int)
gy = ((tx["y_location"] - tx["y_location"].min()) // BIN).astype(int)
nx, ny = gx.max() + 1, gy.max() + 1

def raster(gene=None):
    m = np.ones(len(tx), bool) if gene is None else (tx["feature_name"] == gene).to_numpy()
    img = np.zeros((ny, nx))
    np.add.at(img, (gy[m], gx[m]), 1)
    return img

total = raster()
picks = [g for g in ["EPCAM", "COL1A1", "PTPRC"] if g in set(tx["feature_name"])][:3]

fig, axes = plt.subplots(1, len(picks) + 1, figsize=(4.4 * (len(picks) + 1), 4.4))
axes[0].imshow(total, cmap="magma", vmax=np.percentile(total, 99))
axes[0].set_title(f"all transcripts, {BIN} um bins")
for ax, g in zip(axes[1:], picks):
    im = raster(g)
    ax.imshow(im, cmap="magma", vmax=max(np.percentile(im, 99.5), 1))
    ax.set_title(g)
for ax in axes:
    ax.axis("off")
plt.tight_layout(); plt.show()

> **Try it yourself — how coarse is too coarse?**
>
> `BIN` sets the grid square size in microns. Small bins are noisy; large bins blur
> structure away. A typical cell is 10–20 µm across — what happens when your bin is
> smaller than a cell, and when it is much bigger?

In [ ]:
TRY_BIN = 10        # <-- CHANGE THIS (try 3, 10, 25, 50)

gx2 = ((tx["x_location"] - tx["x_location"].min()) // TRY_BIN).astype(int)
gy2 = ((tx["y_location"] - tx["y_location"].min()) // TRY_BIN).astype(int)
img2 = np.zeros((gy2.max() + 1, gx2.max() + 1))
np.add.at(img2, (gy2, gx2), 1)

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.imshow(img2, cmap="magma", vmax=np.percentile(img2, 99))
ax.axis("off"); ax.set_title(f"{TRY_BIN} um bins")
plt.show()

print(f"grid is {img2.shape[1]} x {img2.shape[0]} squares")
print(f"median transcripts per square: {np.median(img2[img2 > 0]):.0f}")

Compare these images with your cell-type map. **If a spatial pattern is visible in
the raw transcript density but absent from the cell-type map, your segmentation ate
it** — often the case for cells with little cytoplasm, or for regions where the
boundary stain failed.

This is the cheapest sanity check in the whole field and almost nobody does it.

### Exercise 5.3
Compute the correlation between two genes on the 10 um grid, and again between the
same two genes across cells. Do they agree? A pair that correlates on the grid but
not across cells is a candidate for a segmentation-driven artefact — or for a
genuine paracrine relationship between adjacent cells, which is exactly the
ambiguity you have to think through.

In [ ]:
# your code here

## 4. Where this leaves you

| Question | scRNA-seq | Xenium |
|---|---|---|
| Which cell types are present? | **better** — whole transcriptome, deeper | limited to the panel |
| Rare cell state discovery | **better** | panel has to already contain the markers |
| Which cells touch which? | impossible | direct |
| Expression vs distance to a structure | impossible | direct |
| Tissue compartments / niches | impossible | direct |
| Dissociation-sensitive cell types | badly biased | unbiased |
| Sample size | many cells, cheap | expensive per section, **n is the number of sections** |

The honest summary: **they answer different questions and the best studies use both.**
Discovery in dissociated data, placement and validation in situ. Anyone telling you
one replaces the other is selling something.

---

That is the workshop. What you have seen in these five notebooks is the core of what
spatial data lets you do that dissociated data does not.

**Before you plan your own study**, read `docs/DESIGNING_YOUR_STUDY.md`. It is the
checklist we did not have time for: whether your question is actually spatial, what
*n* means when cells are not replicates, whether the panel contains your answer, and
what to budget for storage and compute. The commonest way to waste a Xenium run is to
decide these things afterwards.